# Perform a Layer by Layer forward pass on a model

This notebook is purely for debugging purposes. It is not meant to be run as a script. The goal is to perform a forward pass on a model layer by layer, and print the output of each layer, and compare it to the output of the RISC-V/C model.

TODO: Clean this up later, this is a mess right now.

In [1]:
import os
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import random
import csv

from sklearn.model_selection import train_test_split

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

In [2]:
# Load and preprocess MNIST dataset
(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0  # Normalize the images to [0, 1]
x_train = np.expand_dims(x_train, -1)  # Add channel dimension
y_train = tf.keras.utils.to_categorical(y_train, 10)  # One-hot encode the labels

In [33]:
# Get a 1 random image
idx = random.randint(0, len(x_train) - 1)
image = x_train[idx].squeeze()  # (28, 28)
label = np.argmax(y_train[idx])  # Get label

with open("mnist_single_sample.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    
    # Flatten the image and combine it with the label
    row = [label] + image.flatten().tolist()
    
    # Write row to CSV
    writer.writerow(row)

print(label)

2


In [67]:
import numpy as np

def print_output_shape_and_values(x):
    print(f"Output shape: {x.shape}")
    
    # If it's a 4D tensor (e.g., batch of images), handle it
    if len(x.shape) == 4:
        _, height, width, channels = x.shape
        for c in range(channels):
            for i in range(height):
                for j in range(width):
                    print(f"{x[0, i, j, c]:.3f}", end=" ")
                print()
            print()
        
    
    # If it's a 2D array (after flattening), handle it
    elif len(x.shape) == 2:
        # Extract height and width from flattened shape, assume one image
        rows, cols = x.shape
        for i in range(rows):
            for j in range(cols):
                print(f"{x[i, j]:.3f}", end=" ")
            print()
    else:
        print("Unsupported shape")


## Load the model from mnist_cnn_model.keras

In [45]:
model = tf.keras.models.load_model("mnist_cnn_model.keras")

In [101]:
x = tf.expand_dims(image, axis=0)  # Add batch dimension
x = tf.expand_dims(x, axis=-1)     # Add channel dimension

x = model.layers[0](x)  # First layer output

print_output_shape_and_values(x)

# Flatten the output and reshape it into 8 grids of 24x24
out = x.numpy().flatten()  # Flatten the output

# Reshape into 8 grids of 24x24
grids = out.reshape(8, 24, 24)

# Write to file with nine decimal places
with open("conv2d_out.txt", "w") as f:
    for i in range(24):  # Iterate over rows
        for grid in grids:  # Iterate over each grid
            row_values = ",".join(f"{grid[i, j]:.9f}" for j in range(24))  # Get the ith row of the grid with 9 decimal places
            f.write(row_values + ",")
        # Add a newline after each row from all grids
        # f.write(

Output shape: (1, 24, 24, 8)
-0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.425 -0.311 -0.225 -0.134 -0.071 -0.132 -0.138 -0.246 -0.367 -0.497 -0.572 -0.510 -0.496 -0.460 -0.460 -0.460 -0.460 
-0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.264 -0.048 -0.053 -0.032 0.013 0.113 0.249 0.356 0.398 0.383 0.296 0.009 -0.355 -0.474 -0.534 -0.477 -0.460 -0.460 
-0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.288 -0.184 -0.165 -0.047 0.030 0.036 0.064 0.127 0.255 0.439 0.555 0.524 0.260 -0.141 -0.346 -0.500 -0.477 -0.460 
-0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.531 -0.738 -0.822 -0.630 -0.538 -0.522 -0.554 -0.546 -0.512 -0.333 0.027 0.279 0.362 0.181 -0.104 -0.318 -0.481 -0.460 
-0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.728 -0.899 -0.844 -0.659 -0.725 -0.690 -0.765 -0.732 -0.865 -0.945 -0.726 -0.350 -0.005 0.011 -0.124 -0.176 -0.416 -0.460 
-0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.535 -0.470 -0.448 -0.293 -0.524 -0.578 -0.581 -0.581 -0.658 -0.760 -0.679 -0.577 -0.273 -0.102 -0.160 -

In [102]:
# Second Layer
x = model.layers[1](x)  # Second layer output
print_output_shape_and_values(x)

Output shape: (1, 24, 24, 8)
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.013 0.113 0.249 0.356 0.398 0.383 0.296 0.009 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.030 0.036 0.064 0.127 0.255 0.439 0.555 0.524 0.260 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.027 0.279 0.362 0.181 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.011 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000

In [103]:
# Third Layer
x = model.layers[2](x)  # Third layer output
print_output_shape_and_values(x)

Output shape: (1, 12, 12, 8)
0.000 0.000 0.000 0.000 0.000 0.113 0.356 0.398 0.296 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.036 0.127 0.439 0.555 0.362 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.011 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.035 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.149 0.120 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.278 0.000 0.000 
0.000 0.032 0.077 0.432 0.503 0.471 0.083 0.000 0.154 0.206 0.000 0.000 
0.000 0.303 0.317 0.000 0.038 0.355 0.114 0.034 0.060 0.000 0.000 0.000 
0.000 0.634 0.566 0.155 0.000 0.000 0.123 0.429 0.177 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.149 0.237 0.070 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 

0.178 0.178 0.178 0.822 1.764 3.036 3.570 3.626 3.079 1.686 0.709 0.178 
0.178 0.178 0.178 1.0

In [97]:
# Fourth Layer
x = model.layers[3](x)  # Fourth layer output
print_output_shape_and_values(x)

Output shape: (1, 1152)
0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.822 0.000 0.000 0.000 0.202 0.275 0.000 0.000 1.764 0.000 0.000 0.415 0.241 0.630 0.000 0.113 3.036 0.000 0.000 0.254 0.000 0.718 0.000 0.356 3.570 0.000 0.000 0.204 0.000 0.889 0.000 0.398 3.626 0.000 0.000 0.000 0.000 0.852 0.000 0.296 3.079 0.101 0.000 0.000 0.000 0.620 0.000 0.000 1.686 0.000 0.121 0.000 0.000 0.000 0.000 0.000 0.709 0.000 0.219 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 1.021 0.000 0.000 0.000 0.799 0.000 0.109 0.000 2.015 0.985 0.000 0.222 0.829 0.579 0.000 0.036 2.670 1.169 0.000 0.000 0.000 0.521 0.615 0.127 2.700 0.869 0.000 0.000 0.000 1.016 1.312 0.439 3.068 0.519 0.000 0.000 0.000 1.642 1.243 0.555 3.275 0.24

In [85]:
# Fifth Layer: Dense Layer
x = model.layers[4](x)  # Fifth layer output
print_output_shape_and_values(x)

Output shape: (1, 10)
-13.567 -16.879 14.582 -7.632 -18.014 -19.317 -9.460 -17.614 -2.533 -10.817 


In [58]:
# Sixth Layer: Softmax Layer
x = model.layers[5](x)  # Sixth layer output
print_output_shape_and_values(x)

Output shape: (1, 10)
0.000 0.000 1.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 


In [59]:
print(f"Predicted class: {np.argmax(x)}")
print(f"True class: {label}")

Predicted class: 2
True class: 2
